# Ensemble Agreement as a Calibrated Confidence Signal

Best-of-N deep research on BrowseComp, with agreement measured as a confidence
signal. All logic lives in the `.py` modules beside this notebook; every cell
here is an import plus a call.

**This notebook spends money.** Step 2 is a spend check on a handful of
questions that extrapolates the full cost before you commit. Read its output
before running step 3.

Pipeline: benchmark -> spend check -> full run -> grade -> aggregate ->
calibrate -> sweep -> figures.

In [ ]:
import sys, os, json, asyncio, warnings
from pathlib import Path
from collections import defaultdict
from dataclasses import asdict
from types import SimpleNamespace

sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore", category=FutureWarning)

if not (Path.cwd() / "config.py").exists():
    raise SystemExit(
        f"config.py not found in {Path.cwd()} -- "
        "launch Jupyter from the deep-research directory, or cd into it first."
    )

import numpy as np
import matplotlib.pyplot as plt

from config import (settings, ensure_dirs, ROOT, PRICING, SEARCH_COST_USD,
                    is_free, provider, estimate_days)

# --- the four dials -------------------------------------------------------
BENCHMARK   = "simpleqa"   # "simpleqa" (free-model friendly) or "browsecomp"
N_QUESTIONS = 150          # questions sampled
N_MAX       = 5            # researchers executed per question
SEED        = 0
SPEND_CHECK = 3            # questions to price before the full run

settings.benchmark = BENCHMARK
settings.n_max, settings.n_questions, settings.split_seed = N_MAX, N_QUESTIONS, SEED
ensure_dirs()

# --- credentials, per configured provider --------------------------------
KEY_FOR = {"openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY",
           "google_genai": "GOOGLE_API_KEY", "groq": "GROQ_API_KEY",
           "ollama": None}
needed = {KEY_FOR.get(provider(m)) for m in
          (settings.researcher_model, settings.grader_model)} - {None}
if settings.search_backend == "tavily":
    needed.add("TAVILY_API_KEY")

missing = sorted(k for k in needed if not os.getenv(k))
if missing:
    raise SystemExit(
        f"Missing {', '.join(missing)} in .env, then restart the kernel.\n"
        "Free options: GOOGLE_API_KEY from aistudio.google.com/apikey, "
        "GROQ_API_KEY from console.groq.com/keys."
    )

free = is_free(settings.researcher_model)
print("project root :", ROOT)
print("benchmark    :", BENCHMARK)
print("researcher   :", settings.researcher_model, "(FREE)" if free else "(paid)")
print("grader       :", settings.grader_model)
print("search       :", settings.search_backend,
      f"(${SEARCH_COST_USD.get(settings.search_backend, 0):.3f}/search)")
print("ensemble N   :", N_MAX, " concurrency:", settings.concurrency)

if free:
    days = estimate_days(N_QUESTIONS, N_MAX)
    print(f"\nFree tier: the budget is TIME, not money.")
    print(f"  ~{N_QUESTIONS * N_MAX} agent runs at ~6 requests each")
    print(f"  ~{days:.1f} day(s) of wall clock against the daily request cap")
    if days > 4:
        print("  -> that is a long time. Lower N_QUESTIONS or N_MAX.")

## 1. Benchmark

BrowseComp ships encrypted to stay out of training corpora: each row carries a
plaintext `canary` that is the XOR password for its `problem` and `answer`.

The calibration/test split is drawn once from `SEED` and frozen. Changing the
seed after seeing test numbers leaks the test set.

In [ ]:
from benchmark import load_questions, save_questions, read_questions

if settings.paths["questions"].exists():
    questions = read_questions()
    print(f"reusing existing split: {len(questions)} questions")
else:
    questions = load_questions(N_QUESTIONS, settings.calib_frac, SEED,
                               name=BENCHMARK)
    save_questions(questions)
    print(f"new split written: {len(questions)} questions")

split_by_q = {q.qid: q.split for q in questions}
qmap = {q.qid: q for q in questions}
print(f"  calib {sum(s=='calib' for s in split_by_q.values())}"
      f"   test {sum(s=='test' for s in split_by_q.values())}")
print("\nexample question:\n ", questions[0].problem[:220], "...")

## 2. Spend check

Run the ensemble on a few questions first and extrapolate. BrowseComp questions
are deliberately hard — many searches, long contexts — so per-question cost
varies a lot with the model you picked. Do not skip this.

In [ ]:
_search_cost = SEARCH_COST_USD.get(settings.search_backend, 0.0)
from cache import SearchCache
from researcher import run_ensemble

cache = SearchCache()
out_path = settings.paths["runs"]

def load_runs():
    by_q = defaultdict(list)
    if out_path.exists():
        with open(out_path) as f:
            for line in f:
                if line.strip():
                    rec = json.loads(line)
                    by_q[rec["qid"]].append(SimpleNamespace(**rec))
    return dict(by_q)

async def execute(subset):
    done = {(r.qid, r.member) for rs in load_runs().values() for r in rs}
    with open(out_path, "a") as f:
        for i, q in enumerate(subset):
            if all((q.qid, m) in done for m in range(N_MAX)):
                continue
            runs = await run_ensemble(q.qid, q.problem, N_MAX, cache)
            for r in runs:
                d = asdict(r); d.pop("raw", None)
                f.write(json.dumps(d) + "\n")
            f.flush()

            # Circuit breaker: an unrecoverable error on the very first
            # question will recur on all 300. Stop after one, not after 2400.
            from researcher import is_fatal
            bad = [r for r in runs if is_fatal(r.error)]
            if bad:
                raise SystemExit(
                    f"Stopping at question {i+1}: all {len(bad)}/{len(runs)} "
                    f"members failed with {bad[0].error.split(':')[0]}.\n"
                    f"  {bad[0].error[:160]}\n"
                    "Run `python check.py` for a live API test.")

            print(f"  {i+1}/{len(subset)}  {q.qid}", end="\r")

await execute(questions[:SPEND_CHECK])

runs_by_q = load_runs()

# --- HARD STOP -----------------------------------------------------------
# A misconfigured key produces runs that "succeed" as records while doing
# nothing: zero tokens, zero cost, empty answers. Without this check the sweep
# completes, reports $0.00, and hands you 2400 blanks. Fail here instead.
from researcher import is_fatal

probe = [r for rs in runs_by_q.values() for r in rs]
failed = [r for r in probe if getattr(r, "error", None)]
fatal  = [r for r in failed if is_fatal(r.error)]

if fatal:
    from collections import Counter
    kinds = Counter(r.error.split(":")[0] for r in fatal)
    raise SystemExit(
        f"{len(fatal)}/{len(probe)} runs failed with unrecoverable errors: "
        f"{dict(kinds)}\n"
        "Nothing will work until this is fixed. Most likely an invalid API key.\n"
        "  1. python check.py            # runs a live API test\n"
        "  2. fix .env, restart the kernel\n"
        "  3. python -c \"from researcher import prune_failed_runs as p; "
        "p('runs/ensemble_runs.jsonl')\"\n"
        "Step 3 matters: resume skips recorded runs, so without it the re-run "
        "does nothing.")

if failed:
    print(f"!! {len(failed)}/{len(probe)} runs errored (recoverable). "
          "Inspect before continuing.")

price_in, price_out = PRICING.get(settings.researcher_model, (0.40, 1.60))
tok_in  = sum(r.input_tokens  for rs in runs_by_q.values() for r in rs)
tok_out = sum(r.output_tokens for rs in runs_by_q.values() for r in rs)
searches = sum(r.billable_searches for rs in runs_by_q.values() for r in rs)
nq = len(runs_by_q)

per_q = (tok_in*price_in/1e6 + tok_out*price_out/1e6 + searches*_search_cost) / max(nq,1)
print(f"\n{nq} questions x {N_MAX} members")
print(f"  tokens in/out : {tok_in:,} / {tok_out:,}")
print(f"  billable searches: {searches}   cache: {cache.stats()}")
print(f"  cost per question: ${per_q:.4f}")
if tok_in == 0 and tok_out == 0:
    raise SystemExit(
        "Zero tokens recorded across every run. The model was never actually "
        "called.\nRun `python check.py` -- it makes a live API call and will "
        "tell you why.")

print(f"\n  ESTIMATED FULL RUN ({N_QUESTIONS} questions): ${per_q*N_QUESTIONS:.2f}")
print("  Cache hits will reduce this. Stop here if that number is too high --")
print("  lower N_QUESTIONS or N_MAX, or pick a cheaper researcher model.")

## 3. Full run

Resumable: completed `(qid, member)` pairs are skipped, so an interruption costs
nothing to restart. This is the long cell — expect hours, not minutes.

In [ ]:
await execute(questions)

runs_by_q = load_runs()

from researcher import is_fatal
_all = [r for rs in runs_by_q.values() for r in rs]
_fatal = [r for r in _all if is_fatal(getattr(r, "error", None))]
if _fatal:
    raise SystemExit(
        f"{len(_fatal)}/{len(_all)} runs hit unrecoverable errors -- see the "
        "spend-check cell for the recovery steps. Do not continue; every cell "
        "below will run happily on empty data and give you zeros.")

complete = {q: rs for q, rs in runs_by_q.items() if len(rs) >= N_MAX}
errors = [r for rs in runs_by_q.values() for r in rs if getattr(r, "error", None)]
print(f"questions with a full ensemble: {len(complete)}/{len(questions)}")
print(f"member-level errors: {len(errors)}")
if errors:
    from collections import Counter
    print("  ", Counter(e.error.split(":")[0] for e in errors).most_common(5))
print(f"search cache: {cache.stats()}")

## 4. Grade

Grade *distinct* `(question, answer)` pairs once and cache the verdict. The
sweep evaluates tens of thousands of subsampled predictions; grading each would
cost more than generating them.

Grader accuracy caps every claim downstream — hand-check ~50 verdicts and report
the agreement rate in your writeup.

In [ ]:
from grader import Grader

grader = Grader()
items, seen = [], set()
for qid, rs in runs_by_q.items():
    if qid not in qmap:
        continue
    for r in rs:
        key = grader._key(qid, r.exact_answer)
        if key in seen or key in grader.cache:
            continue
        seen.add(key)
        items.append((qid, qmap[qid].problem, qmap[qid].answer, r.exact_answer))

print(f"{len(items)} distinct answers need grading")
await grader.grade_all(items)
grade = grader.lookup
print(f"grader calls: {grader.calls}   cache: {len(grader.cache)} verdicts")

## 5. Aggregate into confidence signals

Agreement fraction = share of members in the largest answer cluster. Denominator
is N, not the number of parseable answers: a member that failed is evidence of
difficulty, and dropping it would inflate confidence on exactly the hard
questions.

Ties are broken by seeded coin flip, deliberately. Breaking them by stated
confidence leaks the signal being compared against — measured on the test
fixture it inflated n=2 accuracy from 0.507 to 0.636 and created a spurious
even/odd sawtooth across the sweep.

In [ ]:
from ensemble import aggregate, entropy_confidence

rows = []
for qid, runs in complete.items():
    agg = aggregate(runs, qid, tie_break="random")
    rows.append({
        "qid": qid, "split": split_by_q[qid],
        "agreement": agg.agreement,
        "entropy": entropy_confidence(agg),
        "stated": agg.stated_confidence if agg.stated_confidence is not None else 0.5,
        "correct": float(grade(qid, agg.answer)),
        "is_tie": agg.is_tie,
    })

calib = [r for r in rows if r["split"] == "calib"]
test  = [r for r in rows if r["split"] == "test"]
arr = lambda rs, k: np.array([r[k] for r in rs], float)

print(f"calib {len(calib)}   test {len(test)}")
print(f"plurality accuracy at N={N_MAX} (test): {arr(test,'correct').mean():.3f}")
print(f"mean agreement (test):                  {arr(test,'agreement').mean():.3f}")
print(f"mean stated confidence (test):          {arr(test,'stated').mean():.3f}")
print(f"tie rate (test):                        {arr(test,'is_tie').mean():.3f}")

### Sanity check

If agreement carries no information about correctness there is nothing to
calibrate. Verify that accuracy rises with agreement *before* fitting anything.

In [ ]:
a, y = arr(test, "agreement"), arr(test, "correct")
print(f"{'agreement':>12} {'n':>5} {'accuracy':>10}")
for lo, hi in [(0, .4), (.4, .6), (.6, .8), (.8, 1.01)]:
    m = (a >= lo) & (a < hi)
    if m.sum():
        print(f"{lo:.1f}-{hi:<7.1f} {m.sum():>5} {y[m].mean():>10.3f}")

from scipy.stats import pointbiserialr
r, p = pointbiserialr(y, a)
print(f"\npoint-biserial r = {r:.3f}  (p = {p:.2e})")
if r <= 0:
    print("\n!! agreement carries no signal. Stop and debug before calibrating.")

## 6. Calibration: fit on calib, report on test

A one-dimensional monotone mapping fitted on a held-out split — the same
structure as temperature scaling. Raw agreement is reported alongside because it
is already a probability estimate with **zero** fitted parameters; if it wins
untouched, that is the cleaner result.

Agreement is discrete (N+1 levels), so it is binned by level. Forcing quantile
bins onto 7 distinct values puts several bins on identical values and makes the
reliability curve double back on itself.

In [ ]:
from calibration import (ece, brier, brier_decomposition, fit_calibrator,
                         bootstrap_ci, paired_permutation_test)

iso   = fit_calibrator(arr(calib,"agreement"), arr(calib,"correct"), "isotonic")
platt = fit_calibrator(arr(calib,"agreement"), arr(calib,"correct"), "platt")

signals = {
    "verbalised confidence": arr(test, "stated"),
    "raw agreement":         arr(test, "agreement"),
    "agreement + isotonic":  iso.predict(arr(test, "agreement")),
    "agreement + Platt":     platt.predict(arr(test, "agreement")),
}
y = arr(test, "correct")

print(f"{'signal':<24}{'ECE [95% CI]':>26}{'Brier':>8}{'resol.':>9}{'bins':>10}")
results = {}
for name, p_ in signals.items():
    strat = "quantile" if name.startswith("verbalised") else "levels"
    e, lo, hi = bootstrap_ci(p_, y, ece, n_boot=2000, n_bins=10, strategy=strat)
    d = brier_decomposition(p_, y, 10)
    results[name] = {"ece": e, "ece_lo": lo, "ece_hi": hi, "brier": brier(p_, y),
                     "reliability": d["reliability"], "resolution": d["resolution"],
                     "uncertainty": d["uncertainty"], "bins": strat}
    print(f"{name:<24}{e:>7.3f} [{lo:.3f}, {hi:.3f}]{brier(p_,y):>8.3f}"
          f"{d['resolution']:>9.4f}{strat:>10}")

`resolution` measures whether a signal *discriminates*. A signal that always
reports the base rate is perfectly calibrated and completely useless — near-zero
ECE, zero resolution. Read the two together.

In [ ]:
p_val = paired_permutation_test(signals["raw agreement"],
                                signals["verbalised confidence"],
                                y, ece, n_perm=10000, n_bins=10, strategy="levels")
print(f"agreement vs verbalised, paired permutation on ECE: p = {p_val:.4f}")
print("significant at 0.05" if p_val < 0.05 else
      "NOT significant -- do not claim a difference")

## 7. Cost / quality sweep

`N=8` was run once; smaller ensembles are estimated by drawing random
`n`-subsets from those eight. Cost is modelled as `n x` the per-member mean
rather than read off the `N=8` totals, because a smaller ensemble would have had
a colder search cache — this over-states large-`N` cost, the conservative
direction for a claim that small `N` suffices.

The `ties` column matters: where it is high, the tie-break rule rather than
agreement is deciding the answer.

In [ ]:
_search_cost = SEARCH_COST_USD.get(settings.search_backend, 0.0)
from sweep import sweep, knee

points = sweep(complete, grade, sizes=tuple(range(1, N_MAX+1)), n_draws=200,
               price_in=price_in, price_out=price_out,
               search_cost=_search_cost, seed=SEED)
k = knee(points, z=1.0)

print(f"{'N':>3}{'accuracy':>11}{'+/-':>8}{'agreement':>12}{'ties':>8}{'USD/q':>10}")
for pt in points:
    print(f"{pt.n:>3}{pt.accuracy:>11.3f}{pt.accuracy_se:>8.3f}"
          f"{pt.mean_agreement:>12.3f}{pt.tie_rate:>8.3f}{pt.usd:>10.4f}"
          f"{'  <- knee' if pt.n==k else ''}")

kp, top = next(p for p in points if p.n==k), points[-1]
print(f"\nN={k} reaches {kp.accuracy:.3f} at ${kp.usd:.4f}/question")
print(f"N={top.n} reaches {top.accuracy:.3f} at ${top.usd:.4f}/question")
print(f"-> the last {top.n-k} members buy {top.accuracy-kp.accuracy:+.3f} "
      f"accuracy for {top.usd/kp.usd:.1f}x the cost")

## 8. Figures

Left: reliability. The diagonal is perfect calibration; below it is
overconfident. Right: accuracy against dollars, annotated by ensemble size.

The confidence histogram is not optional — a curve hugging the diagonal means
little if most of the mass sits in one bin.

In [ ]:
from plots import reliability_diagram, cost_frontier, confidence_histogram
from config import FIGS

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.8),
                       gridspec_kw={"width_ratios":[1, 1.15]})
reliability_diagram(signals, y, n_bins=10, strategy="levels", ax=ax[0])
cost_frontier(points, knee_n=k, ax=ax[1])
plt.tight_layout()
fig.savefig(FIGS / "headline.png", dpi=180)
plt.show()

In [ ]:
fig2, ax2 = plt.subplots(figsize=(7, 2.6))
confidence_histogram(signals, ax=ax2)
plt.tight_layout()
fig2.savefig(FIGS / "confidence_histogram.png", dpi=180)
plt.show()

## 9. Save the results table

So the README can quote the numbers without re-running anything.

In [ ]:
from config import RUNS

payload = {
    "n_calib": len(calib), "n_test": len(test), "n_max": N_MAX,
    "researcher_model": settings.researcher_model,
    "grader_model": settings.grader_model,
    "temperature": settings.temperature,
    "tie_break": "random",
    "test_accuracy_at_n_max": float(arr(test,"correct").mean()),
    "calibration": results,
    "permutation_p_agreement_vs_verbalised": p_val,
    "knee": k,
    "sweep": [vars(pt) for pt in points],
}
(RUNS / "results.json").write_text(json.dumps(payload, indent=2))
print(f"wrote {RUNS/'results.json'}")
print(f"figures in {FIGS}")

## What to write up

Open the README with the ECE table and the two figures, then answer three
questions in order:

1. **Is agreement better calibrated than the model's own confidence?** Point
   estimate, interval, permutation p-value. If it is not significant, say so —
   that is still a result, and claiming it anyway is the fastest way to fail an
   interview question.
2. **Does fitting help beyond raw agreement?** If isotonic barely beats raw, the
   honest headline is that a zero-parameter signal is already calibrated, which
   is a stronger claim than one needing a fitted mapping.
3. **Where is the knee, and what does the tail cost?** Phrase it as money per
   accuracy point, not a ratio of N.

Then the caveats: grader accuracy on a hand-checked sample, the retrieval-cache
tradeoff, correlated ensemble errors (members agreeing on the same wrong answer
is the failure mode this approach is blind to), tie rates at even N, and
benchmark contamination.